In [25]:
!pip install ultralytics torchvision
import cv2
import os
import glob
import numpy as np
import torch
import torchvision
from torchvision import transforms
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO

############################################################
# Mathematical Utilities
############################################################

def iou_matrix(boxes1, boxes2):
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)))

    b1_x1, b1_y1, b1_x2, b1_y2 = boxes1[:, 0], boxes1[:, 1], boxes1[:, 2], boxes1[:, 3]
    b2_x1, b2_y1, b2_x2, b2_y2 = boxes2[:, 0], boxes2[:, 1], boxes2[:, 2], boxes2[:, 3]

    inter_x1 = np.maximum(b1_x1[:, None], b2_x1)
    inter_y1 = np.maximum(b1_y1[:, None], b2_y1)
    inter_x2 = np.minimum(b1_x2[:, None], b2_x2)
    inter_y2 = np.minimum(b1_y2[:, None], b2_y2)

    inter_w = np.maximum(0, inter_x2 - inter_x1)
    inter_h = np.maximum(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area1 = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    area2 = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)

    union_area = area1[:, None] + area2 - inter_area
    return inter_area / np.maximum(union_area, 1e-6)

def cosine_distance_matrix(feat1, feat2):
    if len(feat1) == 0 or len(feat2) == 0:
        return np.zeros((len(feat1), len(feat2)))
    feat1 = feat1 / np.maximum(np.linalg.norm(feat1, axis=1, keepdims=True), 1e-6)
    feat2 = feat2 / np.maximum(np.linalg.norm(feat2, axis=1, keepdims=True), 1e-6)
    sim = np.dot(feat1, feat2.T)
    return np.clip(1.0 - sim, 0.0, 2.0)

############################################################
# Feature Extractor (Swin Transformer)
############################################################

class ReIDExtractor:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        print("Loading Pre-trained Swin Transformer (Swin-S) Re-ID Model...")
        self.model = torchvision.models.swin_s(weights=torchvision.models.Swin_S_Weights.IMAGENET1K_V1)
        self.model.head = torch.nn.Identity() # Output is 768-d for Swin-S
        self.model.to(self.device)
        self.model.eval()

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    @torch.no_grad()
    def extract(self, frame, boxes):
        if len(boxes) == 0:
            return np.empty((0, 768))

        crops = []
        for box in boxes:
            x1, y1, x2, y2 = map(float, box[:4])
            w, h = x2 - x1, y2 - y1
            x1, y1 = max(0, int(x1 + w * 0.1)), max(0, int(y1 + h * 0.1))
            x2, y2 = min(frame.shape[1], int(x2 - w * 0.1)), min(frame.shape[0], int(y2 - h * 0.1))

            if x2 <= x1 or y2 <= y1:
                crop_tensor = torch.zeros((3, 224, 224))
            else:
                crop = frame[y1:y2, x1:x2]
                crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                crop_tensor = self.transform(crop)
            crops.append(crop_tensor)

        batch = torch.stack(crops).to(self.device)
        features = self.model(batch)
        return features.cpu().numpy()

############################################################
# True Kalman Filter & Track Entity
############################################################

class AdvancedKalmanBoxTracker:
    count = 1

    def __init__(self, bbox, feature=None):
        self.x = np.zeros((10, 1))
        self.x[:4, 0] = self._convert_bbox_to_z(bbox)
        self.P = np.eye(10) * 10.0
        self.P[4:, 4:] *= 1000.0
        self.F = np.eye(10)
        for i in range(4): self.F[i, i+4] = 1.0
        self.F[0, 8], self.F[1, 9] = 0.5, 0.5
        self.F[4, 8], self.F[5, 9] = 1.0, 1.0
        self.H = np.zeros((4, 10))
        for i in range(4): self.H[i, i] = 1.0
        self.R = np.eye(4) * 10.0
        self.Q = np.eye(10) * 0.1

        self.id = AdvancedKalmanBoxTracker.count
        AdvancedKalmanBoxTracker.count += 1
        self.time_since_update = 0
        self.hits = 1

        self.smooth_feat = feature

    def _convert_bbox_to_z(self, bbox):
        w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
        return np.array([bbox[0] + w / 2., bbox[1] + h / 2., w, h])

    def _convert_x_to_bbox(self):
        x, y, w, h = self.x[:4, 0]
        return np.array([x - w/2., y - h/2., x + w/2., y + h/2.])

    def predict(self):
        self.x = np.dot(self.F, self.x)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q
        self.time_since_update += 1
        return self._convert_x_to_bbox()

    def update(self, bbox, feature=None, det_conf=0.0):
        self.time_since_update = 0
        self.hits += 1

        if feature is not None and det_conf > 0.60:
            if self.smooth_feat is None:
                self.smooth_feat = feature
            else:
                self.smooth_feat = 0.9 * self.smooth_feat + 0.1 * feature
                self.smooth_feat /= np.maximum(np.linalg.norm(self.smooth_feat), 1e-6)

        z = self._convert_bbox_to_z(bbox).reshape(4, 1)
        S = np.dot(self.H, np.dot(self.P, self.H.T)) + self.R
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        y = z - np.dot(self.H, self.x)
        self.x = self.x + np.dot(K, y)
        self.P = np.dot((np.eye(10) - np.dot(K, self.H)), self.P)

############################################################
# Fast Two-Stage Tracker (Core BoT-SORT Logic)
############################################################

class SimpleBoTSORT:
    def __init__(self, high_thresh=0.30, low_thresh=0.05, max_age=30, combined_match_thresh=0.75):
        self.high_thresh = high_thresh
        self.low_thresh = low_thresh
        self.max_age = max_age
        self.match_thresh = combined_match_thresh
        self.tracks = []

    def update(self, detections, features):
        predicted_boxes = np.array([t.predict() for t in self.tracks]) if len(self.tracks) > 0 else np.empty((0, 4))
        trk_feats = np.array([t.smooth_feat if t.smooth_feat is not None else np.zeros(768) for t in self.tracks]) if len(self.tracks) > 0 else np.empty((0, 768))

        if len(detections) > 0:
            high_mask = detections[:, 4] >= self.high_thresh
            low_mask = (detections[:, 4] > self.low_thresh) & (detections[:, 4] < self.high_thresh)

            high_dets, high_feats = detections[high_mask], features[high_mask]
            low_dets = detections[low_mask]
        else:
            high_dets, low_dets = np.empty((0, 5)), np.empty((0, 5))
            high_feats = np.empty((0, 768))

        # Stage 1: Match high-confidence detections
        matched, unmatched_trks, unmatched_dets = self._match(predicted_boxes, high_dets[:, :4], trk_feats, high_feats, use_reid=True)
        for trk_idx, det_idx in matched:
            self.tracks[trk_idx].update(high_dets[det_idx][:4], high_feats[det_idx], det_conf=high_dets[det_idx, 4])

        # Stage 2: Match low-confidence detections
        active_unmatched_trks = [t for t in unmatched_trks if self.tracks[t].time_since_update == 1]
        if len(active_unmatched_trks) > 0 and len(low_dets) > 0:
            rem_pred_boxes = predicted_boxes[active_unmatched_trks]
            matched_low, _, _ = self._match(rem_pred_boxes, low_dets[:, :4], None, None, use_reid=False)
            for trk_idx_local, det_idx in matched_low:
                trk_idx = active_unmatched_trks[trk_idx_local]
                self.tracks[trk_idx].update(low_dets[det_idx][:4], feature=None, det_conf=low_dets[det_idx, 4])

        # Initialize new tracks
        for det_idx in unmatched_dets:
            self.tracks.append(AdvancedKalmanBoxTracker(high_dets[det_idx][:4], high_feats[det_idx]))

        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]
        return self.tracks

    def _match(self, trk_boxes, det_boxes, trk_feats, det_feats, use_reid=True):
        if len(trk_boxes) == 0 or len(det_boxes) == 0:
            return [], list(range(len(trk_boxes))), list(range(len(det_boxes)))

        ious = iou_matrix(trk_boxes, det_boxes)
        iou_cost = 1.0 - ious

        gate_mask = ious < 0.05

        if use_reid:
            reid_cost = cosine_distance_matrix(trk_feats, det_feats)

            # THE OFFICIAL BOT-SORT MATH:
            theta_e = 0.25
            theta_iou = 0.50

            gated_reid_cost = np.ones_like(reid_cost)
            valid_mask = (reid_cost < theta_e) & (iou_cost < theta_iou)
            gated_reid_cost[valid_mask] = 0.5 * reid_cost[valid_mask]

            # Take the minimum of spatial cost and gated re-id cost
            cost_matrix = np.minimum(iou_cost, gated_reid_cost)
            match_thresh = self.match_thresh
        else:
            cost_matrix = iou_cost
            match_thresh = 0.85

        cost_matrix[gate_mask] = 10.0

        trk_indices, det_indices = linear_sum_assignment(cost_matrix)

        matched, unmatched_trks, unmatched_dets = [], [], []
        for t, d in zip(trk_indices, det_indices):
            if cost_matrix[t, d] > match_thresh:
                unmatched_trks.append(t)
                unmatched_dets.append(d)
            else:
                matched.append((t, d))

        unmatched_trks.extend(list(set(range(len(trk_boxes))) - set(trk_indices)))
        unmatched_dets.extend(list(set(range(len(det_boxes))) - set(det_indices)))

        return matched, unmatched_trks, unmatched_dets

############################################################
# Main Application (I/O Structure)
############################################################

class CustomBotSORTGenerator:
    def __init__(self, image_dir="Images4", output_file="ref_botsort_submission.txt", model_name="yolo26x.pt"):
        self.image_dir = image_dir
        self.output_file = output_file
        self.model_name = model_name

        self.model = None
        self.reid = None

        self.tracker = SimpleBoTSORT(high_thresh=0.30, low_thresh=0.05, max_age=30, combined_match_thresh=0.75)

    def load_models(self):
        print("=" * 60)
        self.model = YOLO(self.model_name)
        print(f"Loaded YOLO Detector: {self.model_name}")
        self.reid = ReIDExtractor()
        print("Loaded Appearance Network.")
        print("=" * 60)

    def get_image_list(self):
        image_paths = sorted(glob.glob(os.path.join(self.image_dir, "*.jpg")))
        if len(image_paths) == 0:
            raise RuntimeError(f"No jpg images found inside '{self.image_dir}'")
        print(f"Found {len(image_paths)} frames.")
        return image_paths

    @staticmethod
    def write_detection(file, frame, track_id, box):
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        file.write(f"{frame},{track_id},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},-1,-1,-1,-1\n")

    def generate_submission(self):
        self.load_models()
        image_paths = self.get_image_list()
        use_half = torch.cuda.is_available()

        print("Running Gated Advanced Kalman + Re-ID (Swin-S) Custom BotSORT...")

        with open(self.output_file, "w") as out_file:
            for frame_idx, img_path in enumerate(image_paths, start=1):
                frame = cv2.imread(img_path)
                if frame is None: continue

                results = self.model.predict(
                    source=frame, classes=[0], conf=0.05, iou=0.70,
                    imgsz=1280, half=use_half, verbose=False
                )

                result = results[0]
                if result.boxes is None or len(result.boxes) == 0:
                    self.tracker.update(np.empty((0, 5)), np.empty((0, 768)))
                    continue

                boxes = result.boxes.xyxy.cpu().numpy()
                confs = result.boxes.conf.cpu().numpy()
                detections = np.hstack((boxes, confs.reshape(-1, 1)))

                features = self.reid.extract(frame, boxes)

                active_tracks = self.tracker.update(detections, features)

                for track in active_tracks:
                    if track.time_since_update == 0 and track.hits >= 2:
                        self.write_detection(out_file, frame_idx, track.id, track._convert_x_to_bbox())

                if frame_idx % 10 == 0 or frame_idx == len(image_paths):
                    print(f"Processed {frame_idx}/{len(image_paths)} frames")

        print("Tracking Complete. Results saved to", self.output_file)

if __name__ == "__main__":
    tracker = CustomBotSORTGenerator(
        image_dir="Images4",
        output_file="ref_botsort_submission.txt",
        model_name="yolo26x.pt"
    )
    tracker.generate_submission()

Loaded YOLO Detector: yolo26x.pt
Loading Pre-trained Swin Transformer (Swin-S) Re-ID Model...
Loaded Appearance Network.
Found 600 frames.
Running Gated Advanced Kalman + Re-ID (Swin-S) Custom BotSORT...
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. U

In [27]:
!pip install motmetrics

import numpy as np

# ==============================================================================
# NUMPY 2.0 PATCH
# motmetrics uses np.asfarray, which was removed in NumPy 2.0.
# We recreate it here so the library functions normally without needing a downgrade.
# ==============================================================================
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)

import motmetrics as mm
from collections import defaultdict

def load_mot_file(filename):
    """
    Parses a standard MOT format file: frame, id, x, y, w, h
    Returns a dictionary mapping frame_id to 'ids' and 'boxes'.
    """
    data = defaultdict(lambda: {'ids': [], 'boxes': []})

    with open(filename, "r") as f:
        for line in f:
            vals = line.strip().split(",")
            if len(vals) < 6:
                continue

            frame = int(float(vals[0]))
            tid = int(float(vals[1]))
            x = float(vals[2])
            y = float(vals[3])
            w = float(vals[4])
            h = float(vals[5])

            data[frame]['ids'].append(tid)
            data[frame]['boxes'].append([x, y, w, h])

    return dict(data)

def evaluate(gt_file, pred_file, iou_threshold=0.5):
    """
    Evaluates tracking performance using the official motmetrics library.
    """
    print("=" * 60)
    print("Loading Data and Computing Distance Matrices...")
    print("=" * 60)

    gt = load_mot_file(gt_file)
    pred = load_mot_file(pred_file)

    # Initialize the official MOT accumulator
    acc = mm.MOTAccumulator(auto_id=True)

    frames = sorted(set(gt.keys()) | set(pred.keys()))

    for frame in frames:
        gt_frame = gt.get(frame, {'ids': [], 'boxes': []})
        pred_frame = pred.get(frame, {'ids': [], 'boxes': []})

        gt_ids = gt_frame['ids']
        gt_boxes = np.array(gt_frame['boxes']) if len(gt_frame['boxes']) > 0 else np.empty((0, 4))

        pred_ids = pred_frame['ids']
        pred_boxes = np.array(pred_frame['boxes']) if len(pred_frame['boxes']) > 0 else np.empty((0, 4))

        # Calculate the standard 1 - IoU distance matrix
        # motmetrics natively sets distances > max_iou to NaN to forbid invalid matches
        max_distance = 1.0 - iou_threshold
        dist_matrix = mm.distances.iou_matrix(gt_boxes, pred_boxes, max_iou=max_distance)

        # Update the accumulator (handles ID tracking and switching internally)
        acc.update(gt_ids, pred_ids, dist_matrix)

    print("Calculating Metrics...\n")

    # Generate the metrics summary
    mh = mm.metrics.create()
    summary = mh.compute(
        acc,
        metrics=[
            'mota', 'idf1', 'idp', 'idr', 'precision', 'recall',
            'num_objects', 'num_matches', 'num_switches',
            'num_false_positives', 'num_misses'
        ],
        name='Tracker'
    )

    # Format and print the results clearly
    formatter_map = {
        'idf1': 'IDF1',
        'idp': 'IDP',
        'idr': 'IDR',
        'mota': 'MOTA',
        'num_objects': 'GT_Boxes',
        'num_matches': 'True_Positives',
        'num_switches': 'ID_Switches',
        'num_false_positives': 'False_Positives',
        'num_misses': 'False_Negatives',
        'precision': 'Precision',
        'recall': 'Recall'
    }

    str_summary = mm.io.render_summary(
        summary,
        formatters=mh.formatters,
        namemap=formatter_map
    )

    print(str_summary)

if __name__ == "__main__":
    evaluate("ref/gt.txt", "ref_botsort_submission.txt")

Loading Data and Computing Distance Matrices...
Calculating Metrics...

         MOTA  IDF1   IDP   IDR Precision Recall GT_Boxes True_Positives ID_Switches False_Positives False_Negatives
Tracker 25.7% 25.0% 36.3% 19.0%     76.1%  39.9%    29193          11166         481            3664           17546
